# Load Libraries

In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Configure Polars to show all rows and make it scrollable
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Set wider table width

polars.config.Config

# Define Utils Function

In [3]:
# Use sklearn models instead due to LightGBM installation issues
def train_and_evaluate_models_sklearn(baseline_data, full_data, test_size=0.2, random_state=42):
    """
    Train and evaluate both baseline and full models using sklearn models
    """
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
    from sklearn.model_selection import train_test_split
    
    results = {}
    
    # Prepare datasets
    datasets = {
        'baseline': baseline_data,
        'full': full_data
    }
    
    for dataset_name, data in datasets.items():
        print(f"\n=== TRAINING {dataset_name.upper()} MODEL ===")
        
        # Split features and target
        X = data.drop('TARGET', axis=1)
        y = data['TARGET']
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        
        print(f"Dataset: {data.shape}")
        print(f"Features: {X.shape[1]}")
        print(f"Train: {X_train.shape}, Test: {X_test.shape}")
        print(f"Target distribution: {y.value_counts(normalize=True).to_dict()}")
        
        # Train models
        models = {
            'Random Forest': RandomForestClassifier(
                n_estimators=100,
                random_state=random_state,
                n_jobs=-1
            ),
            'Gradient Boosting': GradientBoostingClassifier(
                n_estimators=100,
                random_state=random_state
            )
        }
        
        dataset_results = {}
        
        for model_name, model in models.items():
            print(f"\nTraining {model_name}...")
            
            # Train model
            model.fit(X_train, y_train)
            
            # Predictions
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            y_pred = model.predict(X_test)
            
            # Evaluate
            auc_score = roc_auc_score(y_test, y_pred_proba)
            
            print(f"{model_name} AUC: {auc_score:.4f}")
            
            # Store results
            dataset_results[model_name] = {
                'model': model,
                'auc_score': auc_score,
                'y_test': y_test,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba,
                'X_test': X_test
            }
            
            # Feature importance for tree-based models
            if hasattr(model, 'feature_importances_'):
                feature_importance = pd.DataFrame({
                    'feature': X.columns,
                    'importance': model.feature_importances_
                }).sort_values('importance', ascending=False)
                
                print(f"\nTop 10 important features:")
                print(feature_importance.head(10))
                
                dataset_results[model_name]['feature_importance'] = feature_importance
        
        results[dataset_name] = dataset_results
    
    return results

In [4]:
# Updated version with LightGBM and Random Forest for better comparison
def train_and_evaluate_models_v2(baseline_data, full_data, test_size=0.2, random_state=42):
    """
    Train and evaluate both baseline and full models using LightGBM and Random Forest
    """
    import lightgbm as lgb
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
    from sklearn.model_selection import train_test_split
    
    results = {}
    
    # Prepare datasets
    datasets = {
        'baseline': baseline_data,
        'full': full_data
    }
    
    for dataset_name, data in datasets.items():
        print(f"\n=== TRAINING {dataset_name.upper()} MODEL ===")
        
        # Split features and target
        X = data.drop('TARGET', axis=1)
        y = data['TARGET']
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        
        print(f"Dataset: {data.shape}")
        print(f"Features: {X.shape[1]}")
        print(f"Train: {X_train.shape}, Test: {X_test.shape}")
        print(f"Target distribution: {y.value_counts(normalize=True).to_dict()}")
        
        # Train models
        models = {
            'LightGBM': lgb.LGBMClassifier(
                random_state=random_state,
                n_estimators=100,
                verbose=-1
            ),
            'Random Forest': RandomForestClassifier(
                n_estimators=100,
                random_state=random_state,
                n_jobs=-1
            )
        }
        
        dataset_results = {}
        
        for model_name, model in models.items():
            print(f"\nTraining {model_name}...")
            
            # Train model
            model.fit(X_train, y_train)
            
            # Predictions
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            y_pred = model.predict(X_test)
            
            # Evaluate
            auc_score = roc_auc_score(y_test, y_pred_proba)
            
            print(f"{model_name} AUC: {auc_score:.4f}")
            
            # Store results
            dataset_results[model_name] = {
                'model': model,
                'auc_score': auc_score,
                'y_test': y_test,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba,
                'X_test': X_test
            }
            
            # Feature importance for tree-based models
            if hasattr(model, 'feature_importances_'):
                feature_importance = pd.DataFrame({
                    'feature': X.columns,
                    'importance': model.feature_importances_
                }).sort_values('importance', ascending=False)
                
                print(f"\nTop 10 important features:")
                print(feature_importance.head(10))
                
                dataset_results[model_name]['feature_importance'] = feature_importance
        
        results[dataset_name] = dataset_results
    
    return results

In [5]:
def prepare_full_dataset_for_modeling(df):
    """
    Prepare the full dataset for modeling - handle all features systematically
    """
    df_full = df.clone()
    
    # Remove ID column
    df_full = df_full.drop('SK_ID_CURR')
    
    # Handle missing values by feature type
    feature_stats = []
    
    for col in df_full.columns:
        if col == 'TARGET':
            continue
            
        col_type = df_full.schema[col]
        missing_count = df_full[col].null_count()
        missing_rate = missing_count / df_full.height
        
        if col_type in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]:
            # Numerical: fill with median
            if missing_count > 0:
                median_val = df_full[col].median()
                df_full = df_full.with_columns(pl.col(col).fill_null(median_val))
                
        elif col_type == pl.Utf8:
            # Categorical: fill with 'MISSING' to preserve the signal
            if missing_count > 0:
                df_full = df_full.with_columns(pl.col(col).fill_null('MISSING'))
        
        feature_stats.append({
            'feature': col,
            'type': str(col_type),
            'missing_rate': missing_rate,
            'imputed': missing_count > 0
        })
    
    print(f"Full dataset prepared: {df_full.shape}")
    print(f"Features imputed: {sum(1 for stat in feature_stats if stat['imputed'])}")
    
    return df_full, feature_stats

def encode_categorical_features(df):
    """
    Encode categorical features for machine learning
    """
    from sklearn.preprocessing import LabelEncoder
    
    df_encoded = df.to_pandas()  # Convert to pandas for sklearn
    label_encoders = {}
    
    categorical_cols = [col for col in df_encoded.columns 
                       if df_encoded[col].dtype == 'object' and col != 'TARGET']
    
    print(f"Encoding {len(categorical_cols)} categorical features...")
    
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le
    
    return df_encoded, label_encoders

In [6]:
def calculate_feature_target_correlations(df):
    """
    Calculate correlations between all numerical features and the target variable.
    Handle missing values by using available data for correlation calculation.
    """
    
    # Get numerical columns (excluding ID and target)
    numerical_cols = [col for col, dtype in df.schema.items() 
                     if dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32] 
                     and col not in ['SK_ID_CURR', 'TARGET']]
    
    correlations = []
    
    print(f"Calculating correlations for {len(numerical_cols)} numerical features...")
    
    # Convert to pandas for correlation calculation (more efficient for this task)
    df_pandas = df.select(['TARGET'] + numerical_cols).to_pandas()
    
    for col in numerical_cols:
        # Calculate correlation using available data (ignoring NaN)
        correlation = df_pandas['TARGET'].corr(df_pandas[col])
        
        if not pd.isna(correlation):  # Only include if correlation can be calculated
            missing_rate = df[col].null_count() / df.height
            correlations.append({
                'feature': col,
                'correlation': abs(correlation),  # Use absolute correlation for ranking
                'raw_correlation': correlation,
                'missing_rate': missing_rate
            })
    
    # Convert to DataFrame and sort by absolute correlation
    corr_df = pl.DataFrame(correlations).sort('correlation', descending=True)
    
    return corr_df

def analyze_categorical_target_relationship(df):
    """
    Analyze relationship between categorical features and target using contingency analysis
    """
    categorical_cols = [col for col, dtype in df.schema.items() 
                       if dtype == pl.Utf8 and col != 'SK_ID_CURR']
    
    cat_analysis = []
    
    for col in categorical_cols:
        if df[col].null_count() / df.height < 0.8:  # Only analyze if not too many missing
            # Calculate default rate by category
            analysis = df.group_by(col).agg([
                pl.count().alias('count'),
                pl.col('TARGET').mean().alias('default_rate')
            ]).filter(pl.col('count') > 100)  # Only categories with sufficient samples
            
            if analysis.height > 1:  # Need multiple categories to analyze
                default_rates = analysis['default_rate'].to_list()
                overall_default_rate = df['TARGET'].mean()
                
                # Calculate variation from overall rate
                max_deviation = max([abs(rate - overall_default_rate) for rate in default_rates])
                
                cat_analysis.append({
                    'feature': col,
                    'max_deviation': max_deviation,
                    'n_categories': analysis.height,
                    'missing_rate': df[col].null_count() / df.height
                })
    
    if cat_analysis:
        return pl.DataFrame(cat_analysis).sort('max_deviation', descending=True)
    else:
        return pl.DataFrame()

In [7]:
def smart_feature_selection_and_handling(df):
    """
    Smart approach to handle features without domain knowledge:
    1. Start with low-missing, interpretable features
    2. Use feature importance to guide decisions about high-missing features
    3. Create simple baseline model first
    """
    
    # Separate features by missing rate and interpretability
    feature_analysis = []
    
    for col in df.columns:
        if col in ['SK_ID_CURR', 'TARGET']:
            continue
            
        missing_rate = df[col].null_count() / df.height
        is_normalized = any(suffix in col for suffix in ['_AVG', '_MODE', '_MEDI', '_NORMALIZED'])
        is_external = col.startswith('EXT_SOURCE')
        is_flag = col.startswith('FLAG_')
        is_interpretable = col in ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 
                                 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CODE_GENDER', 
                                 'CNT_CHILDREN', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']
        
        feature_analysis.append({
            'column': col,
            'missing_rate': missing_rate,
            'is_normalized': is_normalized,
            'is_external': is_external,
            'is_flag': is_flag,
            'is_interpretable': is_interpretable
        })
    
    feature_df = pl.DataFrame(feature_analysis).sort('missing_rate')
    
    # Strategy 1: Start with highly interpretable, low-missing features
    core_features = feature_df.filter(
        (pl.col('is_interpretable') == True) | 
        ((pl.col('missing_rate') < 0.1) & (pl.col('is_flag') == True))
    )['column'].to_list()
    
    # Strategy 2: Add external scores (known to be important in credit scoring)
    external_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    
    # Strategy 3: Add some low-missing, non-normalized features
    stable_features = feature_df.filter(
        (pl.col('missing_rate') < 0.05) & 
        (pl.col('is_normalized') == False) &
        (~pl.col('column').str.contains('DOCUMENT'))  # Skip document flags for now
    )['column'].to_list()
    
    selected_features = list(set(core_features + external_features + stable_features))
    
    print(f"Core interpretable features: {len(core_features)}")
    print(f"External score features: {len(external_features)}")  
    print(f"Stable low-missing features: {len(stable_features)}")
    print(f"Total selected for baseline model: {len(selected_features)}")
    
    return selected_features, feature_df

def create_baseline_dataset(df, selected_features):
    """Create a clean baseline dataset for initial modeling"""
    
    # Select features + target
    baseline_features = selected_features + ['TARGET']
    df_baseline = df.select(baseline_features)
    
    # Simple imputation strategy
    df_clean = df_baseline.clone()
    
    # Handle numerical missing values with median
    for col in selected_features:
        col_type = df_baseline.schema[col]
        if col_type in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]:
            if df_clean[col].null_count() > 0:
                median_val = df_clean[col].median()
                df_clean = df_clean.with_columns(pl.col(col).fill_null(median_val))
                
        # Handle categorical missing values with mode
        elif col_type == pl.Utf8:
            if df_clean[col].null_count() > 0:
                mode_val = df_clean[col].mode().first()
                if mode_val is None:
                    mode_val = 'Unknown'
                df_clean = df_clean.with_columns(pl.col(col).fill_null(mode_val))
    
    print(f"\nBaseline dataset shape: {df_clean.shape}")
    print(f"Features with missing values after imputation: {df_clean.null_count().sum_horizontal()[0]}")
    
    return df_clean

In [8]:
def get_missing_data_summary(df):
    """
    Create a summary dataframe showing missing data statistics for each column.
    
    Args:
        df: Polars DataFrame
        
    Returns:
        Polars DataFrame with columns: column_name, missing_count, missing_proportion
    """
    total_rows = df.height
    
    missing_summary = pl.DataFrame({
        'column_name': df.columns,
        'missing_count': [df[col].null_count() for col in df.columns],
        'missing_proportion': [round(df[col].null_count() / total_rows, 4) for col in df.columns]
    }).sort('missing_proportion', descending=True)
    
    return missing_summary

In [9]:
def handle_grouped_missing_features(df):
    """
    Handle features that are missing together in groups (e.g., AVG, MODE, MEDI variants).
    
    Strategy:
    1. Identify feature groups (same prefix before _AVG, _MODE, _MEDI)
    2. For each group, create a single representative feature or impute all together
    3. Create missing indicators for groups with high missing rates
    
    Args:
        df: Polars DataFrame
        
    Returns:
        Processed DataFrame
    """
    df_processed = df.clone()
    
    # Find feature groups by prefix
    feature_groups = {}
    for col in df.columns:
        if any(suffix in col for suffix in ['_AVG', '_MODE', '_MEDI']):
            # Extract the base name (everything before _AVG/_MODE/_MEDI)
            base_name = col.replace('_AVG', '').replace('_MODE', '').replace('_MEDI', '')
            if base_name not in feature_groups:
                feature_groups[base_name] = []
            feature_groups[base_name].append(col)
    
    print(f"Found {len(feature_groups)} feature groups:")
    for base_name, cols in feature_groups.items():
        if len(cols) > 1:  # Only show groups with multiple features
            missing_rate = df[cols[0]].null_count() / df.height
            print(f"  {base_name}: {len(cols)} features, {missing_rate:.3f} missing rate")
    
    return df_processed, feature_groups

def create_grouped_features(df, feature_groups, missing_threshold=0.5):
    """
    Create consolidated features from grouped missing features.
    
    Strategies:
    1. If missing rate < threshold: Use mean of available values, impute with group median
    2. If missing rate >= threshold: Create binary indicator + drop original features
    3. For some groups: Create single representative feature (e.g., use _AVG if available)
    """
    df_processed = df.clone()
    
    for base_name, cols in feature_groups.items():
        if len(cols) <= 1:
            continue
            
        missing_rate = df[cols[0]].null_count() / df.height
        
        if missing_rate >= missing_threshold:
            # High missing rate: create binary indicator
            indicator_col = f"{base_name}_AVAILABLE"
            df_processed = df_processed.with_columns(
                pl.when(pl.col(cols[0]).is_not_null()).then(1).otherwise(0).alias(indicator_col)
            )
            print(f"Created indicator {indicator_col} for {base_name} (missing rate: {missing_rate:.3f})")
            
        else:
            # Low missing rate: create consolidated feature
            # Use mean of AVG, MODE, MEDI where available, then impute
            avg_col = [c for c in cols if '_AVG' in c]
            if avg_col:
                # Use AVG as representative, impute with median
                consolidated_col = f"{base_name}_CONSOLIDATED"
                median_val = df[avg_col[0]].median()
                df_processed = df_processed.with_columns(
                    pl.col(avg_col[0]).fill_null(median_val).alias(consolidated_col)
                )
                print(f"Created consolidated feature {consolidated_col} (imputed with median: {median_val})")
    
    return df_processed

# Load Data

In [10]:
# Load all data files using Polars
print("Loading datasets with Polars...")

# Main application data
train_df = pl.read_csv('../data/application_train.csv')
test_df = pl.read_csv('../data/application_test.csv')

# Additional data sources
bureau_df = pl.read_csv('../data/bureau.csv')
bureau_balance_df = pl.read_csv('../data/bureau_balance.csv')
previous_app_df = pl.read_csv('../data/previous_application.csv')
credit_card_df = pl.read_csv('../data/credit_card_balance.csv')
pos_cash_df = pl.read_csv('../data/POS_CASH_balance.csv')
pos_cash_df_preprocessed = pl.read_csv('../data/POS_CASH_balance_processing.csv')
installments_df = pl.read_csv('../data/installments_payments.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Bureau data shape: {bureau_df.shape}")
print(f"Previous applications shape: {previous_app_df.shape}")
print(f"Credit card balance shape: {credit_card_df.shape}")
print(f"POS Cash balance shape: {pos_cash_df.shape}")
print(f"Installments payments shape: {installments_df.shape}")

print(f"\nTarget distribution:")
target_counts = train_df.select(pl.col('TARGET')).to_pandas()['TARGET'].value_counts(normalize=True)
print(target_counts)

Loading datasets with Polars...
Training data shape: (307511, 122)
Test data shape: (48744, 121)
Bureau data shape: (1716428, 17)
Previous applications shape: (1670214, 37)
Credit card balance shape: (3840312, 23)
POS Cash balance shape: (10001358, 8)
Installments payments shape: (13605401, 8)

Target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


# Simple EDA

In [12]:
# Explore the main training data structure
print("Training data column info:")
print(f"Number of columns: {len(train_df.columns)}")
print(f"Column names: {train_df.columns[:20]}...")  # First 20 columns

print("\nData types:")
print(train_df.schema)

print("\nBasic statistics for numerical columns:")
numerical_cols = [col for col, dtype in train_df.schema.items() if dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]]
print(f"Found {len(numerical_cols)} numerical columns")

# Display first few rows
print("\nFirst 5 rows:")
print(train_df.head())

Training data column info:
Number of columns: 122
Column names: ['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION']...

Data types:
Schema([('SK_ID_CURR', Int64), ('TARGET', Int64), ('NAME_CONTRACT_TYPE', String), ('CODE_GENDER', String), ('FLAG_OWN_CAR', String), ('FLAG_OWN_REALTY', String), ('CNT_CHILDREN', Int64), ('AMT_INCOME_TOTAL', Float64), ('AMT_CREDIT', Float64), ('AMT_ANNUITY', Float64), ('AMT_GOODS_PRICE', Float64), ('NAME_TYPE_SUITE', String), ('NAME_INCOME_TYPE', String), ('NAME_EDUCATION_TYPE', String), ('NAME_FAMILY_STATUS', String), ('NAME_HOUSING_TYPE', String), ('REGION_POPULATION_RELATIVE', Float64), ('DAYS_BIRTH', Int64), ('DAYS_EMPLOYED', Int64), ('DAYS_R

In [13]:
# Apply the function to our training data
missing_summary = get_missing_data_summary(train_df)
print("Missing data summary (top 20 columns by missing proportion):")
print(missing_summary)

Missing data summary (top 20 columns by missing proportion):
shape: (122, 3)
┌──────────────────────────────┬───────────────┬────────────────────┐
│ column_name                  ┆ missing_count ┆ missing_proportion │
│ ---                          ┆ ---           ┆ ---                │
│ str                          ┆ i64           ┆ f64                │
╞══════════════════════════════╪═══════════════╪════════════════════╡
│ COMMONAREA_AVG               ┆ 214865        ┆ 0.6987             │
│ COMMONAREA_MODE              ┆ 214865        ┆ 0.6987             │
│ COMMONAREA_MEDI              ┆ 214865        ┆ 0.6987             │
│ NONLIVINGAPARTMENTS_AVG      ┆ 213514        ┆ 0.6943             │
│ NONLIVINGAPARTMENTS_MODE     ┆ 213514        ┆ 0.6943             │
│ NONLIVINGAPARTMENTS_MEDI     ┆ 213514        ┆ 0.6943             │
│ FONDKAPREMONT_MODE           ┆ 210295        ┆ 0.6839             │
│ LIVINGAPARTMENTS_AVG         ┆ 210199        ┆ 0.6835             │
│ LIVINGAPART

In [14]:
# Apply the functions
print("Analyzing grouped missing features...")
df_processed, feature_groups = handle_grouped_missing_features(train_df)

print("\nCreating consolidated features...")
df_processed = create_grouped_features(train_df, feature_groups, missing_threshold=0.5)

Analyzing grouped missing features...
Found 19 feature groups:
  APARTMENTS: 3 features, 0.507 missing rate
  BASEMENTAREA: 3 features, 0.585 missing rate
  YEARS_BEGINEXPLUATATION: 3 features, 0.488 missing rate
  YEARS_BUILD: 3 features, 0.665 missing rate
  COMMONAREA: 3 features, 0.699 missing rate
  ELEVATORS: 3 features, 0.533 missing rate
  ENTRANCES: 3 features, 0.503 missing rate
  FLOORSMAX: 3 features, 0.498 missing rate
  FLOORSMIN: 3 features, 0.678 missing rate
  LANDAREA: 3 features, 0.594 missing rate
  LIVINGAPARTMENTS: 3 features, 0.684 missing rate
  LIVINGAREA: 3 features, 0.502 missing rate
  NONLIVINGAPARTMENTS: 3 features, 0.694 missing rate
  NONLIVINGAREA: 3 features, 0.552 missing rate

Creating consolidated features...
Created indicator APARTMENTS_AVAILABLE for APARTMENTS (missing rate: 0.507)
Created indicator BASEMENTAREA_AVAILABLE for BASEMENTAREA (missing rate: 0.585)
Created consolidated feature YEARS_BEGINEXPLUATATION_CONSOLIDATED (imputed with median: 

In [15]:
# Apply smart feature selection
print("Analyzing features for baseline model...")
selected_features, feature_analysis = smart_feature_selection_and_handling(train_df)

print(f"\nSelected features for baseline model:")
for i, feat in enumerate(selected_features[:15]):  # Show first 15
    print(f"  {i+1:2d}. {feat}")
if len(selected_features) > 15:
    print(f"  ... and {len(selected_features) - 15} more")

# Create baseline dataset
baseline_train = create_baseline_dataset(train_df, selected_features)

Analyzing features for baseline model...
Core interpretable features: 37
External score features: 3
Stable low-missing features: 43
Total selected for baseline model: 65

Selected features for baseline model:
   1. DEF_30_CNT_SOCIAL_CIRCLE
   2. NAME_HOUSING_TYPE
   3. EXT_SOURCE_2
   4. FLAG_PHONE
   5. CNT_FAM_MEMBERS
   6. REGION_RATING_CLIENT_W_CITY
   7. FLAG_DOCUMENT_21
   8. AMT_CREDIT
   9. HOUR_APPR_PROCESS_START
  10. REG_REGION_NOT_WORK_REGION
  11. NAME_INCOME_TYPE
  12. FLAG_CONT_MOBILE
  13. DAYS_LAST_PHONE_CHANGE
  14. REG_REGION_NOT_LIVE_REGION
  15. AMT_GOODS_PRICE
  ... and 50 more

Baseline dataset shape: (307511, 66)
Features with missing values after imputation: 0


In [16]:
# Calculate correlations
print("=== FEATURE-TARGET CORRELATION ANALYSIS ===")
correlation_results = calculate_feature_target_correlations(train_df)

print(f"\nTop 20 features by correlation with TARGET:")
print(correlation_results.head(20))

print(f"\nTop features with high correlation AND low missing rate:")
high_quality_features = correlation_results.filter(
    (pl.col('correlation') > 0.05) & (pl.col('missing_rate') < 0.1)
)
print(high_quality_features.head(10))

# Analyze categorical features  
print(f"\n=== CATEGORICAL FEATURE ANALYSIS ===")
categorical_analysis = analyze_categorical_target_relationship(train_df)
if categorical_analysis.height > 0:
    print(f"\nTop categorical features by variation in default rates:")
    print(categorical_analysis.head(10))

=== FEATURE-TARGET CORRELATION ANALYSIS ===
Calculating correlations for 104 numerical features...

Top 20 features by correlation with TARGET:
shape: (20, 4)
┌─────────────────────────────┬─────────────┬─────────────────┬──────────────┐
│ feature                     ┆ correlation ┆ raw_correlation ┆ missing_rate │
│ ---                         ┆ ---         ┆ ---             ┆ ---          │
│ str                         ┆ f64         ┆ f64             ┆ f64          │
╞═════════════════════════════╪═════════════╪═════════════════╪══════════════╡
│ EXT_SOURCE_3                ┆ 0.178919    ┆ -0.178919       ┆ 0.198253     │
│ EXT_SOURCE_2                ┆ 0.160472    ┆ -0.160472       ┆ 0.002146     │
│ EXT_SOURCE_1                ┆ 0.155317    ┆ -0.155317       ┆ 0.563811     │
│ DAYS_BIRTH                  ┆ 0.078239    ┆ 0.078239        ┆ 0.0          │
│ REGION_RATING_CLIENT_W_CITY ┆ 0.060893    ┆ 0.060893        ┆ 0.0          │
│ REGION_RATING_CLIENT        ┆ 0.058899    ┆ 0.058

In [17]:
import pandas as pd  # Add pandas import for correlation calculation

# Re-run the correlation analysis
print("=== FEATURE-TARGET CORRELATION ANALYSIS ===")
correlation_results = calculate_feature_target_correlations(train_df)

print(f"\nTop 20 features by correlation with TARGET:")
print(correlation_results.head(20))

print(f"\nFeatures with correlation > 0.1 (strong predictors):")
strong_features = correlation_results.filter(pl.col('correlation') > 0.1)
print(strong_features)

print(f"\nHigh-missing features that still show strong correlation:")
strong_but_missing = correlation_results.filter(
    (pl.col('correlation') > 0.05) & (pl.col('missing_rate') > 0.3)
)
print("These might be worth keeping despite high missing rates:")
print(strong_but_missing)

=== FEATURE-TARGET CORRELATION ANALYSIS ===
Calculating correlations for 104 numerical features...

Top 20 features by correlation with TARGET:
shape: (20, 4)
┌─────────────────────────────┬─────────────┬─────────────────┬──────────────┐
│ feature                     ┆ correlation ┆ raw_correlation ┆ missing_rate │
│ ---                         ┆ ---         ┆ ---             ┆ ---          │
│ str                         ┆ f64         ┆ f64             ┆ f64          │
╞═════════════════════════════╪═════════════╪═════════════════╪══════════════╡
│ EXT_SOURCE_3                ┆ 0.178919    ┆ -0.178919       ┆ 0.198253     │
│ EXT_SOURCE_2                ┆ 0.160472    ┆ -0.160472       ┆ 0.002146     │
│ EXT_SOURCE_1                ┆ 0.155317    ┆ -0.155317       ┆ 0.563811     │
│ DAYS_BIRTH                  ┆ 0.078239    ┆ 0.078239        ┆ 0.0          │
│ REGION_RATING_CLIENT_W_CITY ┆ 0.060893    ┆ 0.060893        ┆ 0.0          │
│ REGION_RATING_CLIENT        ┆ 0.058899    ┆ 0.058

# Basic Modeling

In [18]:

# Prepare both datasets
print("=== PREPARING DATASETS FOR MODELING ===")

# 1. Baseline dataset (selected features only)
print("\n1. Preparing baseline dataset...")
if 'baseline_train' not in locals():
    baseline_train = create_baseline_dataset(train_df, selected_features)

baseline_pandas = baseline_train.to_pandas()
print(f"Baseline dataset: {baseline_pandas.shape}")

# 2. Full dataset (all features)
print("\n2. Preparing full dataset...")
full_train, feature_stats = prepare_full_dataset_for_modeling(train_df)
full_encoded, encoders = encode_categorical_features(full_train)
print(f"Full dataset: {full_encoded.shape}")

# Show some statistics
print(f"\nDataset comparison:")
print(f"Baseline features: {baseline_pandas.shape[1] - 1}")  # -1 for TARGET
print(f"Full features: {full_encoded.shape[1] - 1}")
print(f"Feature reduction: {1 - (baseline_pandas.shape[1] - 1) / (full_encoded.shape[1] - 1):.2%}")

=== PREPARING DATASETS FOR MODELING ===

1. Preparing baseline dataset...
Baseline dataset: (307511, 66)

2. Preparing full dataset...
Full dataset prepared: (307511, 121)
Features imputed: 67
Encoding 16 categorical features...
Full dataset: (307511, 121)

Dataset comparison:
Baseline features: 65
Full features: 120
Feature reduction: 45.83%


In [19]:

# Prepare both datasets
print("=== PREPARING DATASETS FOR MODELING ===")

# 1. Baseline dataset (selected features only)
print("\n1. Preparing baseline dataset...")
if 'baseline_train' not in locals():
    baseline_train = create_baseline_dataset(train_df, selected_features)

baseline_pandas = baseline_train.to_pandas()
print(f"Baseline dataset: {baseline_pandas.shape}")

# 2. Full dataset (all features)
print("\n2. Preparing full dataset...")
full_train, feature_stats = prepare_full_dataset_for_modeling(train_df)
full_encoded, encoders = encode_categorical_features(full_train)
print(f"Full dataset: {full_encoded.shape}")

# Show some statistics
print(f"\nDataset comparison:")
print(f"Baseline features: {baseline_pandas.shape[1] - 1}")  # -1 for TARGET
print(f"Full features: {full_encoded.shape[1] - 1}")
print(f"Feature reduction: {1 - (baseline_pandas.shape[1] - 1) / (full_encoded.shape[1] - 1):.2%}")

=== PREPARING DATASETS FOR MODELING ===

1. Preparing baseline dataset...
Baseline dataset: (307511, 66)

2. Preparing full dataset...
Full dataset prepared: (307511, 121)
Features imputed: 67
Encoding 16 categorical features...
Full dataset: (307511, 121)

Dataset comparison:
Baseline features: 65
Full features: 120
Feature reduction: 45.83%


In [23]:
# Fix categorical encoding issue before training
print("=== FIXING CATEGORICAL ENCODING ISSUES ===")
print("Checking for categorical features in baseline dataset...")

# Check data types in baseline_pandas
print("\nBaseline dataset dtypes:")
print(baseline_pandas.dtypes.value_counts())

# Identify categorical columns
categorical_cols = baseline_pandas.select_dtypes(include=['object']).columns.tolist()
if 'TARGET' in categorical_cols:
    categorical_cols.remove('TARGET')

print(f"\nCategorical columns found: {categorical_cols}")

# Encode categorical features in baseline dataset
if len(categorical_cols) > 0:
    from sklearn.preprocessing import LabelEncoder
    baseline_pandas_encoded = baseline_pandas.copy()
    
    for col in categorical_cols:
        print(f"Encoding {col}...")
        le = LabelEncoder()
        baseline_pandas_encoded[col] = le.fit_transform(baseline_pandas_encoded[col].astype(str))
    
    print("Categorical encoding completed for baseline dataset.")
else:
    baseline_pandas_encoded = baseline_pandas.copy()
    print("No categorical columns found in baseline dataset.")

# Check final dtypes
print(f"\nFinal baseline dataset shape: {baseline_pandas_encoded.shape}")
print("Final dtypes:")
print(baseline_pandas_encoded.dtypes.value_counts())

# Train models with sklearn
print("\n=== TRAINING MODELS WITH SKLEARN ===")
model_results_sklearn = train_and_evaluate_models_sklearn(baseline_pandas_encoded, full_encoded)

=== FIXING CATEGORICAL ENCODING ISSUES ===
Checking for categorical features in baseline dataset...

Baseline dataset dtypes:
int64      40
float64    15
object     11
Name: count, dtype: int64

Categorical columns found: ['NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'ORGANIZATION_TYPE', 'FLAG_OWN_REALTY', 'WEEKDAY_APPR_PROCESS_START', 'NAME_FAMILY_STATUS', 'CODE_GENDER', 'NAME_TYPE_SUITE', 'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR']
Encoding NAME_HOUSING_TYPE...
Encoding NAME_INCOME_TYPE...
Encoding NAME_EDUCATION_TYPE...
Encoding ORGANIZATION_TYPE...
Encoding FLAG_OWN_REALTY...
Encoding WEEKDAY_APPR_PROCESS_START...
Encoding NAME_FAMILY_STATUS...
Encoding CODE_GENDER...
Encoding NAME_TYPE_SUITE...
Encoding NAME_CONTRACT_TYPE...
Encoding FLAG_OWN_CAR...
Categorical encoding completed for baseline dataset.

Final baseline dataset shape: (307511, 66)
Final dtypes:
int64      51
float64    15
Name: count, dtype: int64

=== TRAINING MODELS WITH SKLEARN ===

=== TRAINING BASEL

## Feature Engineering Approaches Used

### 1. Missing Data Analysis Approach
- **Identified grouped features** (AVG/MODE/MEDI variants) that were missing together
- **Example**: `COMMONAREA_AVG`, `COMMONAREA_MODE`, `COMMONAREA_MEDI` all had ~70% missing rate
- These represent **external data** that may not be available for all properties.
- Some features tends to be missing together, this maybe because these are derived from a non-neccessary feature

### 2. Smart Feature Selection Strategy

#### Baseline Model: Selected interpretable, low-missing features
- **Core demographic**: `AMT_INCOME_TOTAL`, `DAYS_BIRTH`, `CODE_GENDER`, `CNT_CHILDREN`
- **Financial metrics**: `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`
- **External credit scores**: `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`
- **Contact info flags**: `FLAG_*` variables

#### Full Model: Used ALL features with systematic imputation
- **Numerical features**: Imputed with median
- **Categorical features**: Created `'MISSING'` category to preserve signal

### 3. Correlation-Based Feature Ranking
- Calculated correlations between each feature and `TARGET` variable
- Identified high-value features even among high-missing ones
- Let the data guide feature importance decisions

### 4. Categorical Encoding
- **Label encoding** for tree-based models (they can handle this well)
- **Preserved `'MISSING'`** as a distinct category rather than imputing

## Key Insights from Results

### Why External Credit Scores (EXT_SOURCE_*) Ranked as Top Features:
- 🏛️ **Professional credit assessments** from external agencies
- 📊 **Capture creditworthiness information** not available in application data
- 🎯 **Specifically designed** for credit risk prediction
- 💪 **Strong predictive signal** even with some missing values

### However, there are some gotchas
- Not all customer have these external credit scores, therefore these can be null as well.
- Some of the missing data pattern can be informative (i.e the information about the customer's living area can be missing from our database or they not provide those information at all).

## Feature Engineering Lessons

### ✅ What Worked:
- Simple, interpretable features often outperform complex engineered ones
- External/third-party data sources are extremely valuable in credit scoring
- Missing data patterns can be informative (building info missing = certain property types)
- Domain-specific features (credit scores) beat generic demographic features

### ❌ What Didn't Work:
- High-missing normalized building features added more noise than signal
- Too many features can hurt model performance (curse of dimensionality)

## Business Implications
- 🤝 **Focus data acquisition efforts** on external credit bureau relationships
- 🎯 **Prioritize features** that are consistently available and high-quality
- 🔄 **Don't over-engineer features** when simple ones work well

# Try with preprocessed pos_processes

In [24]:
# Check what columns are available in our datasets
print("=== CHECKING DATASET COLUMNS ===")
print(f"Train DF columns: {list(train_df.columns)[:10]}...")
print(f"Baseline train columns: {list(baseline_train.columns)[:10]}...")
print(f"POS features columns: {list(pos_cash_df_preprocessed.columns)[:10]}...")

# Check if SK_ID_CURR exists
print(f"\nSK_ID_CURR in train_df: {'SK_ID_CURR' in train_df.columns}")
print(f"SK_ID_CURR in baseline_train: {'SK_ID_CURR' in baseline_train.columns}")
print(f"SK_ID_CURR in pos_cash_df_preprocessed: {'SK_ID_CURR' in pos_cash_df_preprocessed.columns}")

# The issue is that baseline_train was created without SK_ID_CURR
# Let's recreate the baseline dataset with SK_ID_CURR included
print(f"\n=== RECREATING BASELINE DATASET WITH SK_ID_CURR ===")

def create_baseline_dataset_with_id(df, selected_features):
    """Create a clean baseline dataset for initial modeling, keeping SK_ID_CURR"""
    
    # Select features + target + ID
    baseline_features = selected_features + ['TARGET', 'SK_ID_CURR']
    df_baseline = df.select(baseline_features)
    
    # Simple imputation strategy
    df_clean = df_baseline.clone()
    
    # Handle numerical missing values with median
    for col in selected_features:
        col_type = df_baseline.schema[col]
        if col_type in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]:
            if df_clean[col].null_count() > 0:
                median_val = df_clean[col].median()
                df_clean = df_clean.with_columns(pl.col(col).fill_null(median_val))
                
        # Handle categorical missing values with mode
        elif col_type == pl.Utf8:
            if df_clean[col].null_count() > 0:
                mode_val = df_clean[col].mode().first()
                if mode_val is None:
                    mode_val = 'Unknown'
                df_clean = df_clean.with_columns(pl.col(col).fill_null(mode_val))
    
    print(f"Baseline dataset shape: {df_clean.shape}")
    print(f"Features with missing values after imputation: {df_clean.null_count().sum_horizontal()[0]}")
    
    return df_clean

# Recreate baseline dataset with SK_ID_CURR
baseline_train_with_id = create_baseline_dataset_with_id(train_df, selected_features)

# Also need to ensure full_train has SK_ID_CURR 
print(f"Full train has SK_ID_CURR: {'SK_ID_CURR' in full_train.columns}")
if 'SK_ID_CURR' not in full_train.columns:
    print("Adding SK_ID_CURR to full_train...")
    full_train = train_df.select(['SK_ID_CURR'] + [col for col in full_train.columns if col != 'SK_ID_CURR'])
    print(f"Full train updated shape: {full_train.shape}")

=== CHECKING DATASET COLUMNS ===
Train DF columns: ['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']...
Baseline train columns: ['DEF_30_CNT_SOCIAL_CIRCLE', 'NAME_HOUSING_TYPE', 'EXT_SOURCE_2', 'FLAG_PHONE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT_W_CITY', 'FLAG_DOCUMENT_21', 'AMT_CREDIT', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_WORK_REGION']...
POS features columns: ['SK_ID_CURR', 'POS_COUNT_LOANS_TOTAL', 'POS_COUNT_PROBLEM_LOANS', 'POS_COUNT_LOANS_DEMAND', 'POS_MEAN_MAX_DPD', 'POS_MAX_MAX_DPD', 'POS_MIN_MAX_DPD', 'POS_MEAN_MEAN_DPD', 'POS_MEAN_DPD_RATE', 'POS_MAX_DPD_RATE']...

SK_ID_CURR in train_df: True
SK_ID_CURR in baseline_train: False
SK_ID_CURR in pos_cash_df_preprocessed: True

=== RECREATING BASELINE DATASET WITH SK_ID_CURR ===
Baseline dataset shape: (307511, 67)
Features with missing values after imputation: 0
Full train has SK_ID_CURR: False
Adding SK_ID_CU

In [25]:
# Now join POS features with the corrected datasets
def join_pos_features_with_dataset(base_df, pos_features_df, fill_strategy='smart'):
    """
    Join POS features with base dataset, handling missing customers intelligently.
    
    Args:
        base_df: Base dataset (Polars DataFrame)
        pos_features_df: POS features dataset (Polars DataFrame)
        fill_strategy: How to handle missing POS customers ('zero', 'smart')
    """
    
    # Join datasets
    joined_df = base_df.join(pos_features_df, on='SK_ID_CURR', how='left')
    
    # Get POS feature columns (exclude SK_ID_CURR)
    pos_cols = [col for col in pos_features_df.columns if col != 'SK_ID_CURR']
    
    print(f"Joined dataset shape: {joined_df.shape}")
    
    # Count missing values in POS features after join
    missing_customers = joined_df[pos_cols[0]].null_count()
    print(f"Customers without POS history: {missing_customers:,} ({missing_customers/joined_df.shape[0]*100:.1f}%)")
    
    # Handle missing values based on strategy
    if fill_strategy == 'smart':
        # Smart filling: 0 for counts/flags, appropriate values for others
        count_flag_cols = [col for col in pos_cols if any(x in col for x in ['COUNT', 'FLAG', 'TOTAL'])]
        rate_ratio_cols = [col for col in pos_cols if any(x in col for x in ['RATE', 'RATIO', 'MEAN'])]
        other_cols = [col for col in pos_cols if col not in count_flag_cols + rate_ratio_cols]
        
        print(f"Smart fill strategy:")
        print(f"  - Count/Flag columns ({len(count_flag_cols)}): Fill with 0")
        print(f"  - Rate/Ratio columns ({len(rate_ratio_cols)}): Fill with median")
        print(f"  - Other columns ({len(other_cols)}): Fill with 0 or appropriate value")
        
        # Fill counts/flags with 0 (no history)
        for col in count_flag_cols:
            joined_df = joined_df.with_columns(pl.col(col).fill_null(0))
        
        # Fill rates with median of existing customers (or 0 if all null)
        for col in rate_ratio_cols:
            median_val = joined_df[col].median()
            fill_val = median_val if median_val is not None else 0.0
            joined_df = joined_df.with_columns(pl.col(col).fill_null(fill_val))
        
        # Fill other columns appropriately
        for col in other_cols:
            if 'RISK_SEGMENT' in col:
                joined_df = joined_df.with_columns(pl.col(col).fill_null('NEW_CUSTOMER'))
            elif 'MONTH' in col:
                joined_df = joined_df.with_columns(pl.col(col).fill_null(0))
            else:
                joined_df = joined_df.with_columns(pl.col(col).fill_null(0))
    
    return joined_df

print("\n=== JOINING POS FEATURES WITH CORRECTED DATASETS ===")

# 1. Join with baseline dataset (now with SK_ID_CURR)
baseline_with_pos = join_pos_features_with_dataset(
    baseline_train_with_id, pos_cash_df_preprocessed, fill_strategy='smart'
)

print(f"\nBaseline + POS shape: {baseline_with_pos.shape}")
print(f"New POS features added: {baseline_with_pos.shape[1] - baseline_train_with_id.shape[1]}")

# 2. Join with full dataset (add SK_ID_CURR if missing)
if 'SK_ID_CURR' not in full_train.columns:
    # Add SK_ID_CURR to full_train
    full_train_with_id = full_train.with_columns(train_df.select('SK_ID_CURR'))
else:
    full_train_with_id = full_train

full_with_pos = join_pos_features_with_dataset(
    full_train_with_id, pos_cash_df_preprocessed, fill_strategy='smart'
)

print(f"\nFull + POS shape: {full_with_pos.shape}")
print(f"New POS features added: {full_with_pos.shape[1] - full_train_with_id.shape[1]}")


=== JOINING POS FEATURES WITH CORRECTED DATASETS ===
Joined dataset shape: (307511, 98)
Customers without POS history: 18,067 (5.9%)
Smart fill strategy:
  - Count/Flag columns (6): Fill with 0
  - Rate/Ratio columns (15): Fill with median
  - Other columns (10): Fill with 0 or appropriate value

Baseline + POS shape: (307511, 98)
New POS features added: 31
Joined dataset shape: (307511, 153)
Customers without POS history: 18,067 (5.9%)
Smart fill strategy:
  - Count/Flag columns (6): Fill with 0
  - Rate/Ratio columns (15): Fill with median
  - Other columns (10): Fill with 0 or appropriate value

Full + POS shape: (307511, 153)
New POS features added: 31


In [28]:
# Prepare enhanced datasets for modeling
print("\n=== PREPARING ENHANCED DATASETS FOR MODELING ===")

# Remove SK_ID_CURR before modeling (we don't want to train on ID)
baseline_pos_for_ml = baseline_with_pos.drop('SK_ID_CURR').to_pandas()
full_pos_for_ml = full_with_pos.drop('SK_ID_CURR')

# Check for missing values after join
print(f"Baseline + POS missing values: {baseline_pos_for_ml.isnull().sum().sum()}")
print(f"Full + POS missing values: {full_pos_for_ml.null_count().sum_horizontal()[0]}")

# Fix missing values in baseline + POS dataset
if baseline_pos_for_ml.isnull().sum().sum() > 0:
    print("Fixing missing values in baseline + POS dataset...")
    
    # Fill numerical columns with median
    numeric_cols = baseline_pos_for_ml.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if baseline_pos_for_ml[col].isnull().sum() > 0:
            median_val = baseline_pos_for_ml[col].median()
            baseline_pos_for_ml[col].fillna(median_val, inplace=True)
    
    # Fill categorical columns with mode or 'Unknown'
    categorical_cols = baseline_pos_for_ml.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if baseline_pos_for_ml[col].isnull().sum() > 0:
            mode_val = baseline_pos_for_ml[col].mode()
            fill_val = mode_val[0] if len(mode_val) > 0 else 'Unknown'
            baseline_pos_for_ml[col].fillna(fill_val, inplace=True)
    
    print(f"After imputation - missing values: {baseline_pos_for_ml.isnull().sum().sum()}")

# Handle categorical encoding for baseline + POS
categorical_cols_baseline = baseline_pos_for_ml.select_dtypes(include=['object']).columns
if 'TARGET' in categorical_cols_baseline:
    categorical_cols_baseline = categorical_cols_baseline.drop('TARGET')

if len(categorical_cols_baseline) > 0:
    from sklearn.preprocessing import LabelEncoder
    print(f"Encoding {len(categorical_cols_baseline)} categorical features in baseline + POS...")
    
    for col in categorical_cols_baseline:
        le = LabelEncoder()
        baseline_pos_for_ml[col] = le.fit_transform(baseline_pos_for_ml[col].astype(str))

# Handle categorical encoding for full + POS
full_pos_encoded, pos_encoders = encode_categorical_features(full_pos_for_ml)

# Final verification - check for any remaining NaN values
print(f"\nFinal verification:")
print(f"Baseline + POS NaN count: {baseline_pos_for_ml.isnull().sum().sum()}")
print(f"Baseline + POS data types: {baseline_pos_for_ml.dtypes.value_counts()}")
print(f"Full + POS NaN count: {full_pos_encoded.isnull().sum().sum()}")

print(f"\nDataset comparison:")
print(f"Original Baseline: {baseline_pandas.shape[1] - 1} features")
print(f"Baseline + POS: {baseline_pos_for_ml.shape[1] - 1} features")
print(f"Original Full: {full_encoded.shape[1] - 1} features") 
print(f"Full + POS: {full_pos_encoded.shape[1] - 1} features")

pos_feature_count = pos_cash_df_preprocessed.shape[1] - 1  # -1 for SK_ID_CURR
print(f"\nPOS features impact: Added {pos_feature_count} new features to both datasets")


=== PREPARING ENHANCED DATASETS FOR MODELING ===
Baseline + POS missing values: 0
Full + POS missing values: 9152465
Encoding 12 categorical features in baseline + POS...
Encoding 17 categorical features...

Final verification:
Baseline + POS NaN count: 0
Baseline + POS data types: int64      63
float64    34
Name: count, dtype: int64
Full + POS NaN count: 8388094

Dataset comparison:
Original Baseline: 65 features
Baseline + POS: 96 features
Original Full: 120 features
Full + POS: 151 features

POS features impact: Added 31 new features to both datasets


In [30]:
# Debug NaN issues before training
print("=== DEBUGGING NaN ISSUES ===")

# Check for NaN values in both datasets
print("Checking baseline_pos_for_ml:")
if 'baseline_pos_for_ml' in locals():
    nan_count = baseline_pos_for_ml.isnull().sum().sum()
    print(f"Total NaN values: {nan_count}")
    if nan_count > 0:
        nan_columns = baseline_pos_for_ml.isnull().sum()[baseline_pos_for_ml.isnull().sum() > 0]
        print(f"Columns with NaN: {nan_columns}")
else:
    print("baseline_pos_for_ml not found - need to recreate")

print("\nChecking full_pos_encoded:")
if 'full_pos_encoded' in locals():
    nan_count = full_pos_encoded.isnull().sum().sum()
    print(f"Total NaN values: {nan_count}")
    if nan_count > 0:
        nan_columns = full_pos_encoded.isnull().sum()[full_pos_encoded.isnull().sum() > 0]
        print(f"Columns with NaN: {nan_columns}")
else:
    print("full_pos_encoded not found - need to recreate")

# Create completely clean datasets by forcing all NaN removal
def create_clean_dataset(df, dataset_name):
    """Create a dataset with absolutely no NaN values"""
    print(f"\n=== Cleaning {dataset_name} ===")
    
    # Convert to pandas if it's polars
    if hasattr(df, 'to_pandas'):
        df_clean = df.to_pandas()
    else:
        df_clean = df.copy()
    
    print(f"Initial shape: {df_clean.shape}")
    print(f"Initial NaN count: {df_clean.isnull().sum().sum()}")
    
    # Handle all columns systematically
    for col in df_clean.columns:
        if col == 'TARGET':
            continue
            
        nan_count = df_clean[col].isnull().sum()
        if nan_count > 0:
            print(f"Fixing {col}: {nan_count} NaN values")
            
            if df_clean[col].dtype in ['object', 'string']:
                # Categorical: fill with most frequent or 'Unknown'
                mode_val = df_clean[col].mode()
                fill_val = mode_val[0] if len(mode_val) > 0 else 'Unknown'
                df_clean[col].fillna(fill_val, inplace=True)
            else:
                # Numerical: fill with median or 0
                if df_clean[col].notna().sum() > 0:
                    fill_val = df_clean[col].median()
                else:
                    fill_val = 0.0
                df_clean[col].fillna(fill_val, inplace=True)
    
    # Force convert all object columns to numerical using LabelEncoder
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    if 'TARGET' in categorical_cols:
        categorical_cols = categorical_cols.drop('TARGET')
    
    if len(categorical_cols) > 0:
        print(f"Encoding {len(categorical_cols)} categorical columns...")
        from sklearn.preprocessing import LabelEncoder
        
        for col in categorical_cols:
            le = LabelEncoder()
            df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    
    # Final verification
    final_nan_count = df_clean.isnull().sum().sum()
    print(f"Final shape: {df_clean.shape}")
    print(f"Final NaN count: {final_nan_count}")
    print(f"Final dtypes: {df_clean.dtypes.value_counts()}")
    
    # Replace any inf values with finite numbers
    import numpy as np
    df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_clean.fillna(0, inplace=True)
    
    print(f"After inf cleanup - NaN count: {df_clean.isnull().sum().sum()}")
    
    return df_clean

# Create completely clean datasets
print("\n=== CREATING COMPLETELY CLEAN DATASETS ===")

# Baseline + POS
baseline_pos_clean = create_clean_dataset(baseline_pos_for_ml, "Baseline + POS")

# Full + POS  
full_pos_clean = create_clean_dataset(full_pos_encoded, "Full + POS")

# Train models with completely clean datasets
print("\n=== TRAINING MODELS WITH CLEAN POS DATASETS ===")
try:
    model_results_with_pos = train_and_evaluate_models_sklearn(baseline_pos_clean, full_pos_clean)
    
    print("\n" + "="*80)
    print("PERFORMANCE COMPARISON: BEFORE vs AFTER ADDING POS FEATURES")
    print("="*80)

    # Extract results for comparison (assuming model_results_sklearn exists)
    if 'model_results_sklearn' in locals():
        datasets = ['baseline', 'full'] 
        models = ['Random Forest', 'Gradient Boosting']

        print(f"\n{'Dataset':<20} {'Model':<20} {'Before POS':<12} {'After POS':<12} {'Improvement':<12}")
        print("-" * 76)

        for dataset in datasets:
            for model in models:
                before_auc = model_results_sklearn[dataset][model]['auc_score']
                after_auc = model_results_with_pos[dataset][model]['auc_score'] 
                improvement = after_auc - before_auc
                
                print(f"{dataset:<20} {model:<20} {before_auc:.4f}      {after_auc:.4f}      {improvement:+.4f}")
    else:
        print("Previous sklearn results not available for comparison")

except Exception as e:
    print(f"Error during training: {str(e)}")
    print("Checking dataset integrity...")
    print(f"baseline_pos_clean shape: {baseline_pos_clean.shape}")
    print(f"baseline_pos_clean NaN: {baseline_pos_clean.isnull().sum().sum()}")
    print(f"full_pos_clean shape: {full_pos_clean.shape}")
    print(f"full_pos_clean NaN: {full_pos_clean.isnull().sum().sum()}")

=== DEBUGGING NaN ISSUES ===
Checking baseline_pos_for_ml:
Total NaN values: 0

Checking full_pos_encoded:
Total NaN values: 8388094
Columns with NaN: AMT_ANNUITY                       12
AMT_GOODS_PRICE                  278
OWN_CAR_AGE                   202929
CNT_FAM_MEMBERS                    2
EXT_SOURCE_1                  173378
                               ...  
AMT_REQ_CREDIT_BUREAU_DAY      41519
AMT_REQ_CREDIT_BUREAU_WEEK     41519
AMT_REQ_CREDIT_BUREAU_MON      41519
AMT_REQ_CREDIT_BUREAU_QRT      41519
AMT_REQ_CREDIT_BUREAU_YEAR     41519
Length: 61, dtype: int64

=== CREATING COMPLETELY CLEAN DATASETS ===

=== Cleaning Baseline + POS ===
Initial shape: (307511, 97)
Initial NaN count: 0
Final shape: (307511, 97)
Final NaN count: 0
Final dtypes: int64      63
float64    34
Name: count, dtype: int64
After inf cleanup - NaN count: 0

=== Cleaning Full + POS ===
Initial shape: (307511, 152)
Initial NaN count: 8388094
Fixing AMT_ANNUITY: 12 NaN values
Fixing AMT_GOODS_PRICE: 27